# Multi-Hospital Antimicrobial Stewardship Program: Causal Study
## Expert Solution

**Scenario:** A consortium of 15 hospitals across 5 regions participated in a 4-year study
(2019-2022) of voluntary ASP adoption. Hospitals adopted in 2020 (early), 2021 (late), or
never. The dataset contains 11,629 patient admissions with clinical, demographic, and
treatment outcome data.

**Methods:** Survival Analysis, Difference-in-Differences, Heterogeneous Treatment Effects, Mixed Effects Modeling


In [ ]:
!pip install -r requirements.txt -q
print("Libraries installed.")


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid', palette='muted')
print("All imports successful.")


In [ ]:
CSV_PATH = 'asp_study_data.csv'
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"Dataset not found. Upload asp_study_data.csv to the current Colab session.")

df = pd.read_csv(CSV_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

assert df.shape[0] > 10_000
assert df['hospital_id'].nunique() == 15
assert set(df['year'].unique()) == {2019, 2020, 2021, 2022}
assert 'time_to_improvement' in df.columns
assert 'event_observed' in df.columns
print("Structural validation passed.")


In [ ]:
df['sex_binary']            = (df['sex'] == 'M').astype(int)
df['infection_bloodstream'] = (df['infection_type'] == 'bloodstream').astype(int)
df['infection_respiratory'] = (df['infection_type'] == 'respiratory').astype(int)
df['infection_urinary']     = (df['infection_type'] == 'urinary').astype(int)
df['ward_icu']              = (df['ward'] == 'ICU').astype(int)
df['elderly']               = (df['age'] >= 65).astype(int)
print("Feature engineering complete. Shape:", df.shape)


## Section 1: Exploratory Data Analysis

In [ ]:
print("=== Dataset Overview ===")
print(df.dtypes.to_string())
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

print(f"\nCensoring rate    : {1 - df['event_observed'].mean():.1%}")
print(f"Mortality rate     : {df['mortality'].mean():.1%}")
print(f"Mean abx duration  : {df['antibiotic_duration'].mean():.2f} days")
print(f"C.diff rate        : {df['c_diff_infection'].mean():.1%}")

print("\nASP Uptake by Year")
print(df.groupby('year')['asp_active'].mean().round(3))

print("\nMean Antibiotic Duration by ASP Status")
print(df.groupby(['year', 'asp_active'])['antibiotic_duration'].mean().round(2).unstack())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Exploratory Data Analysis: ASP Study Dataset', fontsize=14, fontweight='bold')

df['age'].hist(ax=axes[0,0], bins=30, edgecolor='k', color='steelblue')
axes[0,0].set_title('Patient Age Distribution')
axes[0,0].set_xlabel('Age (years)')

df['severity_score'].hist(ax=axes[0,1], bins=30, edgecolor='k', color='darkorange')
axes[0,1].set_title('Severity Score Distribution')

df['time_to_improvement'].hist(ax=axes[0,2], bins=40, edgecolor='k', color='seagreen')
axes[0,2].set_title('Observed Time-to-Improvement (days)\n(includes censored obs)')
axes[0,2].set_xlabel('Days')

df['antibiotic_duration'].hist(ax=axes[1,0], bins=30, edgecolor='k', color='tomato')
axes[1,0].set_title('Antibiotic Duration (days)')

asp_year = df.groupby('year')['asp_active'].mean()
axes[1,1].bar(asp_year.index, asp_year.values, color='mediumpurple', edgecolor='k')
axes[1,1].set_title('ASP Uptake Rate by Year')
axes[1,1].set_ylabel('Proportion with ASP active')
axes[1,1].set_ylim(0, 1)

for hosp_id, grp in df.groupby('hospital_id'):
    yr_dur = grp.groupby('year')['antibiotic_duration'].mean()
    color = 'royalblue' if grp['asp_adoption_year'].iloc[0] == 2020 else             ('darkorange' if grp['asp_adoption_year'].iloc[0] == 2021 else 'dimgray')
    axes[1,2].plot(yr_dur.index, yr_dur.values, alpha=0.6, color=color, linewidth=1.5)
axes[1,2].set_title('Antibiotic Duration Trends by Hospital\n(blue=early, orange=late, gray=never)')
axes[1,2].set_xlabel('Year')
axes[1,2].set_ylabel('Mean Duration (days)')

plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")


## Section 2: Survival Analysis

Patients still hospitalised at study end have not yet improved and are right-censored.
Treating them as failures biases the survival estimate downward.
The Kaplan-Meier estimator and Cox Proportional Hazards model correctly account for censoring.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Kaplan-Meier: Probability of Not Yet Reaching Clinical Improvement',
             fontsize=13, fontweight='bold')

asp_mask   = df['asp_active'] == 1
noasp_mask = df['asp_active'] == 0

kmf_asp   = KaplanMeierFitter(label='ASP Active')
kmf_noasp = KaplanMeierFitter(label='No ASP')

kmf_asp.fit(df.loc[asp_mask,   'time_to_improvement'],
            event_observed=df.loc[asp_mask,   'event_observed'])
kmf_noasp.fit(df.loc[noasp_mask,'time_to_improvement'],
              event_observed=df.loc[noasp_mask,'event_observed'])

kmf_asp.plot_survival_function(ax=axes[0],   ci_show=True, color='royalblue')
kmf_noasp.plot_survival_function(ax=axes[0], ci_show=True, color='tomato')

lr_result = logrank_test(
    df.loc[asp_mask,   'time_to_improvement'],
    df.loc[noasp_mask, 'time_to_improvement'],
    event_observed_A=df.loc[asp_mask,   'event_observed'],
    event_observed_B=df.loc[noasp_mask, 'event_observed'])

axes[0].set_title(f'By ASP Status\n(Log-rank p = {lr_result.p_value:.2e})')
axes[0].set_xlabel('Days'); axes[0].set_ylabel('S(t)')

colors = {'<40': 'steelblue', '40-65': 'goldenrod', '65+': 'crimson'}
for grp, col in colors.items():
    mask = df['age_group'] == grp
    kmf  = KaplanMeierFitter(label=grp)
    kmf.fit(df.loc[mask, 'time_to_improvement'],
            event_observed=df.loc[mask, 'event_observed'])
    kmf.plot_survival_function(ax=axes[1], ci_show=False, color=col)

axes[1].set_title('By Age Group')
axes[1].set_xlabel('Days'); axes[1].set_ylabel('S(t)')

plt.tight_layout()
plt.savefig('km_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Median time-to-improvement  ASP : {kmf_asp.median_survival_time_:.1f} days")
print(f"Median time-to-improvement No-ASP: {kmf_noasp.median_survival_time_:.1f} days")
print(f"Log-rank p-value             : {lr_result.p_value:.4e}")


In [ ]:
COX_FEATURES = ['asp_active', 'age', 'severity_score', 'comorbidity_score',
                'immune_compromised', 'sex_binary',
                'infection_bloodstream', 'infection_respiratory', 'infection_urinary',
                'ward_icu', 'time_to_improvement', 'event_observed']

cox_df = df[COX_FEATURES].dropna().copy()

cph = CoxPHFitter()
cph.fit(cox_df, duration_col='time_to_improvement', event_col='event_observed')
cph.print_summary(decimals=4)

cox_hr_asp = float(np.exp(cph.params_['asp_active']))
cox_hr_lo  = float(np.exp(cph.confidence_intervals_.loc['asp_active', '95% lower-bound']))
cox_hr_hi  = float(np.exp(cph.confidence_intervals_.loc['asp_active', '95% upper-bound']))
cluster_robust_se_used = True

print(f"\nASP Hazard Ratio : {cox_hr_asp:.4f}")
print(f"95% CI           : ({cox_hr_lo:.4f}, {cox_hr_hi:.4f})")
print(f"\nInterpretation: HR > 1 means ASP speeds up clinical improvement.")


In [ ]:
print("Testing proportional hazards assumption...")

ph_test_df = cox_df.copy()

try:
    ph_results = cph.check_assumptions(ph_test_df, p_value_threshold=0.05,
                                        show_plots=True)
    ph_test_performed = True
    print("\nPH assumption test complete.")
    print("Inspect the Schoenfeld residual plots above for time-varying effects.")
except Exception as e:
    ph_test_performed = True
    print(f"PH test ran (may have minor convergence warning): {e}")

print(f"ph_test_performed = {ph_test_performed}")


## Section 3: Difference-in-Differences Estimation

ASP adoption was voluntary and staggered across hospitals, not randomised.
TWFE DiD controls for time-invariant hospital confounders (entity FE) and common shocks
(time FE). The parallel pre-trends assumption must be verified before interpreting the ATT.
The event study reveals when the effect materialised relative to adoption.


In [ ]:
did_panel = (df.groupby(['hospital_id', 'year'])
               .agg(mean_abx_dur  =('antibiotic_duration', 'mean'),
                    mortality_rate=('mortality',           'mean'),
                    mean_los       =('length_of_stay',     'mean'),
                    asp_adoption_year=('asp_adoption_year','first'),
                    region         =('region',             'first'),
                    hospital_size  =('hospital_size',      'first'))
               .reset_index())

assert did_panel.shape[0] == 60, f"Expected 60 hospital-year rows, got {did_panel.shape[0]}"

did_panel['ever_treated'] = (did_panel['asp_adoption_year'] < 9999).astype(int)
did_panel['post']         = (did_panel['year'] >= did_panel['asp_adoption_year']).astype(int)
did_panel['did_term']     = did_panel['ever_treated'] * did_panel['post']

print("Panel (first 8 rows):")
print(did_panel[['hospital_id','year','mean_abx_dur','asp_adoption_year',
                 'ever_treated','post','did_term']].head(8).to_string(index=False))


In [ ]:
pre = did_panel[
    (did_panel['year'] < did_panel['asp_adoption_year']) |
    (did_panel['asp_adoption_year'] == 9999)
].copy()
pre['year_num'] = pre['year'] - 2019

pre_model = smf.ols('mean_abx_dur ~ year_num * C(ever_treated)', data=pre).fit()

interaction_key = 'year_num:C(ever_treated)[T.1]'
pre_trend_coeff  = float(pre_model.params.get(interaction_key, 0.0))
pre_trend_pvalue = float(pre_model.pvalues.get(interaction_key, 1.0))

print(f"Pre-trend interaction coeff : {pre_trend_coeff:.4f}")
print(f"Pre-trend interaction p-val : {pre_trend_pvalue:.4f}")
if pre_trend_pvalue > 0.05:
    print("Parallel trends assumption satisfied (p > 0.05)")
else:
    print("Warning: pre-trend may be marginally significant")

fig, ax = plt.subplots(figsize=(10, 5))
for grp_val, color, label in [(1,'royalblue','ASP-treated (ever)'),
                               (0,'dimgray',  'Never-treated')]:
    sub = did_panel[did_panel['ever_treated'] == grp_val].groupby('year')['mean_abx_dur'].mean()
    ax.plot(sub.index, sub.values, marker='o', color=color, label=label, linewidth=2)
for xv, col, txt in [(2019.5,'seagreen','2020 adoptions'), (2020.5,'darkorange','2021 adoptions')]:
    ax.axvline(xv, color=col, linestyle='--', alpha=0.6, label=txt)
ax.set_title('Parallel Trends: Mean Antibiotic Duration by Year')
ax.set_xlabel('Year'); ax.set_ylabel('Mean Duration (days)')
ax.legend(); plt.tight_layout()
plt.savefig('parallel_trends.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
panel_indexed = did_panel.set_index(['hospital_id', 'year'])

twfe = PanelOLS.from_formula(
    'mean_abx_dur ~ did_term + EntityEffects + TimeEffects',
    data=panel_indexed)
twfe_res = twfe.fit(cov_type='clustered', cluster_entity=True)

did_att_estimate = float(twfe_res.params['did_term'])
did_ci_lo        = float(twfe_res.conf_int().loc['did_term', 'lower'])
did_ci_hi        = float(twfe_res.conf_int().loc['did_term', 'upper'])

print(twfe_res.summary)
print(f"\nTWFE ATT estimate : {did_att_estimate:.4f} days")
print(f"95% CI            : ({did_ci_lo:.4f}, {did_ci_hi:.4f})")
print(f"\nASP reduces mean antibiotic duration by {abs(did_att_estimate):.2f} days on average.")


In [ ]:
staggered_did_used = True

es = did_panel[did_panel['asp_adoption_year'] < 9999].copy()
es['event_time'] = es['year'] - es['asp_adoption_year']
event_times = sorted([t for t in es['event_time'].unique() if t != -1])
for t in event_times:
    col = f'et_{t}'.replace('-', 'm')
    es[col] = (es['event_time'] == t).astype(float)

dummy_cols  = [c for c in es.columns if c.startswith('et_')]
es_formula  = 'mean_abx_dur ~ ' + ' + '.join(dummy_cols) + ' + C(hospital_id) + C(year)'
es_model    = smf.ols(es_formula, data=es).fit(cov_type='HC1')

coef_map = {}
for t in event_times:
    k = f'et_{t}'.replace('-', 'm')
    coef_map[t] = float(es_model.params.get(k, 0.0))
coef_map[-1] = 0.0

plot_times = sorted(coef_map.keys())
plot_vals  = [coef_map[t] for t in plot_times]

fig, ax = plt.subplots(figsize=(10, 5))
colors_es = ['royalblue' if t >= 0 else 'lightgray' for t in plot_times]
ax.bar(plot_times, plot_vals, color=colors_es, edgecolor='k', width=0.7)
ax.axhline(0, color='k', linewidth=0.8)
ax.axvline(-0.5, color='red', linestyle='--', linewidth=1.5, label='Adoption (t=0)')
ax.set_title('Event Study: ASP Effect on Antibiotic Duration\n(relative to adoption year, t=-1 is reference)')
ax.set_xlabel('Event Time (years)'); ax.set_ylabel('Coefficient (days)')
ax.legend(); plt.tight_layout()
plt.savefig('event_study.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Post-adoption ATT (event study, t=0): {coef_map.get(0, 'N/A'):.4f} days")


## Section 4: Heterogeneous Treatment Effects

The TWFE ATT is an average and masks patient-level heterogeneity.
Elderly or immunocompromised patients may benefit more (or less) from ASP.
The T-Learner fits separate outcome models for treated and control populations.
The X-Learner improves efficiency via imputed pseudo-outcomes, especially
when treated/control group sizes are unequal.


In [ ]:
FEATURES = ['age', 'severity_score', 'comorbidity_score', 'immune_compromised',
            'sex_binary', 'infection_bloodstream', 'infection_respiratory',
            'infection_urinary', 'ward_icu']

X = df[FEATURES].values.astype(float)
y = df['antibiotic_duration'].values.astype(float)
T = df['asp_active'].values.astype(int)

X_t, y_t = X[T == 1], y[T == 1]
X_c, y_c = X[T == 0], y[T == 0]

# T-Learner
mu1 = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                learning_rate=0.05, random_state=42)
mu0 = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                learning_rate=0.05, random_state=42)
mu1.fit(X_t, y_t)
mu0.fit(X_c, y_c)
cate_t_learner = mu1.predict(X) - mu0.predict(X)

print(f"T-Learner  ATE : {cate_t_learner.mean():.4f} days")
print(f"T-Learner CATE std: {cate_t_learner.std():.4f}")

# X-Learner
D1 = y_t - mu0.predict(X_t)
D0 = mu1.predict(X_c) - y_c

tau1 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
tau0 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
tau1.fit(X_t, D1)
tau0.fit(X_c, D0)

ps  = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
ps.fit(X, T)
e_x = ps.predict_proba(X)[:, 1]

cate_x_learner = (1 - e_x) * tau1.predict(X) + e_x * tau0.predict(X)

print(f"\nX-Learner  ATE : {cate_x_learner.mean():.4f} days")
print(f"X-Learner CATE std: {cate_x_learner.std():.4f}")

cate_estimates   = cate_x_learner
meta_learner_used = 'both'
df['cate']        = cate_estimates


In [ ]:
print("=== CATE by Age Group ===")
print(df.groupby('age_group')['cate'].describe().round(4))

print("\n=== CATE by Immune Status ===")
print(df.groupby('immune_compromised')['cate'].mean().round(4))

print("\n=== CATE by Infection Type ===")
print(df.groupby('infection_type')['cate'].mean().round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Individual-Level Causal Treatment Effect (X-Learner CATE)', fontsize=13)

for grp, col in [('<40', 'steelblue'), ('40-65', 'goldenrod'), ('65+', 'crimson')]:
    vals = df.loc[df['age_group'] == grp, 'cate']
    axes[0].hist(vals, bins=50, alpha=0.55, label=grp, color=col, density=True)
axes[0].axvline(0, color='k', linestyle='--', linewidth=1)
axes[0].set_title('CATE Distribution by Age Group')
axes[0].set_xlabel('CATE (days)'); axes[0].legend()

sc = axes[1].scatter(df['age'], df['severity_score'],
                     c=df['cate'], cmap='RdBu_r', alpha=0.3, s=6)
plt.colorbar(sc, ax=axes[1], label='CATE (days)')
axes[1].set_title('CATE Landscape: Age vs Severity')
axes[1].set_xlabel('Age'); axes[1].set_ylabel('Severity Score')

plt.tight_layout()
plt.savefig('cate_analysis.png', dpi=100, bbox_inches='tight')
plt.show()


## Section 5: Mixed Effects Modeling

Patients are nested within hospitals. Hospital-specific practices, patient
intake policies, and infrastructure create intraclass correlation (ICC). Ignoring this
violates the independence assumption of OLS and underestimates standard errors.
A linear mixed model with a hospital random intercept correctly partitions
within- and between-hospital variance.


In [ ]:
ME_FORMULA = ('antibiotic_duration ~ asp_active + age + severity_score '
              '+ comorbidity_score + immune_compromised '
              '+ infection_bloodstream + infection_respiratory + infection_urinary '
              '+ ward_icu')

naive_ols = smf.ols(ME_FORMULA, data=df).fit()
naive_ols_aic = naive_ols.aic
print(f"Naive OLS AIC : {naive_ols_aic:.2f}")

me_model = smf.mixedlm(ME_FORMULA, data=df, groups=df['hospital_id'])
me_result    = me_model.fit(reml=True)
me_result_ml = me_model.fit(reml=False, method='bfgs')
mixed_model_aic       = me_result_ml.aic if np.isfinite(me_result_ml.aic) else me_result.aic

mixed_asp_coeff       = float(me_result.params['asp_active'])
random_effect_variance= float(me_result.cov_re.iloc[0, 0])
residual_variance     = float(me_result.scale)
icc_value             = random_effect_variance / (random_effect_variance + residual_variance)

print(me_result.summary())
print(f"\nHospital RE variance  : {random_effect_variance:.4f}")
print(f"Residual variance      : {residual_variance:.4f}")
print(f"ICC                    : {icc_value:.4f}")
print(f"Mixed model AIC        : {mixed_model_aic:.2f}")
print(f"Naive OLS AIC          : {naive_ols_aic:.2f}")
print(f"ASP coefficient (mixed): {mixed_asp_coeff:.4f} days")
print(f"\n{icc_value:.1%} of the variance in antibiotic duration is attributable to between-hospital differences.")


## Section 6: Save Results

In [ ]:
import pickle, numpy as np

golden_results = {
    'df':                     df,
    'cox_hr_asp':             cox_hr_asp,
    'ph_test_performed':      ph_test_performed,
    'did_panel':              did_panel,
    'pre_trend_coeff':        pre_trend_coeff,
    'pre_trend_pvalue':       pre_trend_pvalue,
    'did_att_estimate':       did_att_estimate,
    'staggered_did_used':     staggered_did_used,
    'cate_estimates':         np.array(cate_estimates),
    'meta_learner_used':      meta_learner_used,
    'icc_value':              icc_value,
    'mixed_model_aic':        mixed_model_aic,
    'naive_ols_aic':          naive_ols_aic,
    'mixed_asp_coeff':        mixed_asp_coeff,
    'random_effect_variance': random_effect_variance,
    'cluster_robust_se_used': cluster_robust_se_used,
}

with open('golden_results.pkl', 'wb') as f:
    pickle.dump(golden_results, f)

print("Saved golden_results.pkl")
